# M6-T4 — Train & Serialize the URL Random Forest Fallback Model

**Owner:** John Graham Kalluri  
**Depends on:** M6-T1 — frozen splits must be present and checksummed before running.

### Inputs / Outputs
| | |
|---|---|
| **In** | `url_train.csv` / `url_test.csv` — 11 engineered URL features (`URL_FEATS` from harness) |
| **Model** | `RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)` via `url_pipeline(scale=False)` |
| **Out** | `models/url_rf.joblib` — serialized pipeline (no scaler + RF) |
| **S3** | `s3://email-security-pipeline-datasets/models/artifacts/url/url_rf.joblib` |

### Acceptance criteria
- Test F1 ≥ 0.85 (reproduces M5-T3 result)
- Artifact saved and uploaded to S3
- SHA-256 of artifact recorded

## Step 0 — Download frozen splits from S3 (skip if already local)
```bash
mkdir -p data/processed
for f in url_train url_test; do
  aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/${f}.csv \
            data/processed/${f}.csv --profile lab-user
done
# verify checksums
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/splits/SPLITS.sha256 \
          data/processed/SPLITS.sha256 --profile lab-user
cd data/processed && sha256sum -c SPLITS.sha256 --ignore-missing && cd -
```

In [8]:
import hashlib
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'M5'))

S3_ARTIFACT = 's3://email-security-pipeline-datasets/models/artifacts/url/url_rf.joblib'
S3_PROFILE  = 'lab-user'  # change to match your AWS profile, e.g. 'aws-lab', 'deploy-user'

# Resolve data dir: works when launched from project root, notebooks/M6/, or Colab (csv in cwd)
_candidates = [
    Path('data/processed'),        # launched from project root
    Path('../../data/processed'),  # launched from notebooks/M6/
    Path('.'),                     # Colab — files copied to cwd
]
PROC = next((p for p in _candidates if (p / 'url_train.csv').exists()), None)
assert PROC is not None, 'url_train.csv not found — run Step 0 to download the splits'

MODELS = Path('models')
MODELS.mkdir(exist_ok=True)
print(f'Data  : {PROC.resolve()}')
print('Split files present.')

Data  : <PROJECT_ROOT>/data/processed
Split files present.


## Step 1 — Load frozen splits

In [9]:
import pandas as pd
from m5_harness import url_pipeline, URL_FEATS

utr = pd.read_csv(PROC / 'url_train.csv')
ute = pd.read_csv(PROC / 'url_test.csv')
print(f'Train : {len(utr):>7,} rows  |  label dist: {utr["label"].value_counts().to_dict()}')
print(f'Test  : {len(ute):>7,} rows  |  label dist: {ute["label"].value_counts().to_dict()}')
print(f'Features ({len(URL_FEATS)}): {URL_FEATS}')

Train : 512,895 rows  |  label dist: {0: 342464, 1: 170431}
Test  : 128,224 rows  |  label dist: {0: 85616, 1: 42608}
Features (11): ['url_length', 'hostname_length', 'num_dots', 'num_hyphens', 'num_at', 'num_digits', 'num_special_chars', 'has_ip', 'has_https', 'num_subdomains', 'is_shortened']


## Step 2 — Train Random Forest pipeline
Uses the shared M5 harness URL pipeline: no StandardScaler (`scale=False`) + RandomForestClassifier.
300 trees with `n_jobs=-1` parallelises training across all available CPU cores.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = url_pipeline(
    RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42,
    ),
    scale=False,
)

rf.fit(utr[URL_FEATS], utr['label'])
print('Training complete.')

Training complete.


## Step 3 — Evaluate on held-out test split

In [11]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

preds = rf.predict(ute[URL_FEATS])
proba = rf.predict_proba(ute[URL_FEATS])[:, 1]

f1 = f1_score(ute['label'], preds)

print('=== Test-set metrics ===')
print(f'  Accuracy  : {accuracy_score(ute["label"], preds):.4f}')
print(f'  Precision : {precision_score(ute["label"], preds):.4f}')
print(f'  Recall    : {recall_score(ute["label"], preds):.4f}')
print(f'  F1        : {f1:.4f}  (target ≥ 0.85)')
print(f'  ROC-AUC   : {roc_auc_score(ute["label"], proba):.4f}')
print()
print(classification_report(ute['label'], preds, target_names=['benign', 'malicious']))

assert f1 >= 0.85, f'F1 {f1:.4f} below target — investigate before uploading artifact'

=== Test-set metrics ===
  Accuracy  : 0.9119
  Precision : 0.8755
  Recall    : 0.8567
  F1        : 0.8660  (target ≥ 0.85)
  ROC-AUC   : 0.9648

              precision    recall  f1-score   support

      benign       0.93      0.94      0.93     85616
   malicious       0.88      0.86      0.87     42608

    accuracy                           0.91    128224
   macro avg       0.90      0.90      0.90    128224
weighted avg       0.91      0.91      0.91    128224



## Step 4 — Serialize artifact
The full pipeline (RF classifier) is saved as a single joblib file.

In [12]:
import joblib

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

artifact_path = MODELS / 'url_rf.joblib'
joblib.dump(rf, artifact_path)

digest = sha256_file(artifact_path)
size_gb = artifact_path.stat().st_size / 1e9

print(f'Artifact : {artifact_path}  ({size_gb:.2f} GB)')
print(f'SHA-256  : {digest}')

Artifact : models/url_rf.joblib  (1.87 GB)
SHA-256  : 1752d0347c50cc191ced1a145c151c0f63f91f50dd0c8495db526dea6a9f58e5


## Step 5 — Upload artifact to S3
Uploads to the `models/artifacts/url/` path so M6-T6 (inference wrapper) and M6-T7 (artifact registry) can find it alongside the Char-CNN artifacts.

In [14]:
import shutil

aws_cli = shutil.which('aws') or '/usr/local/bin/aws'  # conda envs may not include /usr/local/bin

print(f'Uploading {artifact_path.name} ({size_gb:.2f} GB) — this may take a few minutes...')
result = subprocess.run(
    [aws_cli, 's3', 'cp', str(artifact_path), S3_ARTIFACT, '--profile', S3_PROFILE],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f'Upload complete ✅')
    print(f'Verify: aws s3 ls {S3_ARTIFACT} --profile {S3_PROFILE}')
else:
    print(f'Upload failed ⚠️\n{result.stderr}')
    print(f'Run manually: aws s3 cp {artifact_path} {S3_ARTIFACT} --profile {S3_PROFILE}')

Uploading url_rf.joblib (1.87 GB) — this may take a few minutes...
Upload complete ✅
Verify: aws s3 ls s3://email-security-pipeline-datasets/models/artifacts/url/url_rf.joblib --profile lab-user


## Summary

In [15]:
print('=' * 60)
print('M6-T4 URL RANDOM FOREST SUMMARY')
print('=' * 60)
print(f'Model        : RandomForestClassifier(n_estimators=300, random_state=42)')
print(f'Pipeline     : url_pipeline(scale=False)')
print(f'Features ({len(URL_FEATS):2d}): {URL_FEATS}')
print(f'Train rows   : {len(utr):,}')
print(f'Test F1      : {f1:.4f}  {"✅" if f1 >= 0.85 else "❌ below target"}')
print(f'Artifact     : {artifact_path.name}  ({size_gb:.2f} GB)')
print(f'SHA-256      : {digest}')
print(f'S3           : {S3_ARTIFACT}')
print()
print('Consumed by:')
print('  M6-T6  inference wrapper (fallback when GPU unavailable)')
print('  M6-T7  artifact registry')
print('=' * 60)

M6-T4 URL RANDOM FOREST SUMMARY
Model        : RandomForestClassifier(n_estimators=300, random_state=42)
Pipeline     : url_pipeline(scale=False)
Features (11): ['url_length', 'hostname_length', 'num_dots', 'num_hyphens', 'num_at', 'num_digits', 'num_special_chars', 'has_ip', 'has_https', 'num_subdomains', 'is_shortened']
Train rows   : 512,895
Test F1      : 0.8660  ✅
Artifact     : url_rf.joblib  (1.87 GB)
SHA-256      : 1752d0347c50cc191ced1a145c151c0f63f91f50dd0c8495db526dea6a9f58e5
S3           : s3://email-security-pipeline-datasets/models/artifacts/url/url_rf.joblib

Consumed by:
  M6-T6  inference wrapper (fallback when GPU unavailable)
  M6-T7  artifact registry
